In [10]:
import mikeio
import geopandas as gpd
import numpy as np
from shapely.geometry import Polygon
from shapely.errors import GEOSException

# ------------------ USER INPUTS ------------------
dfsu_file = r"D:\Phd Research\Prototype HD Model\Simulation\Without Reef Scenarios\Climate change condition\5yr_Compound_Flood.m21fm - Result Files\5yr_compound_flood_stat.dfsu"

land_shp = r"D:\Phd Research\GIS\Shape\land_Part_area_utm.zip"

thresh = 0.10  # inundation threshold (m)
fallback_crs = "EPSG:32614"
# -------------------------------------------------

# 1) Open DFSU
ds = mikeio.open(dfsu_file)

print("Available DFSU items:")
for i, item in enumerate(ds.items):
    print(f"{i}: {item}")

# Read first item automatically
# Change items=0 if your flood-depth item is different
dset = ds.read(items=1)
geom = ds.geometry

# 2) Extract element-centered depth values
depth = dset[0].to_numpy().squeeze()

if depth.ndim != 1:
    raise RuntimeError(
        f"Unexpected depth shape: {depth.shape}. Expected 1D element values."
    )

# 3) Build element polygons
element_table = geom.element_table
node_coordinates = geom.node_coordinates[:, :2]
n_nodes = len(node_coordinates)

elem_polys = []
valid_elem_mask = []

for nodes in element_table:
    nodes = np.asarray(nodes)

    try:
        nodes = nodes.astype(int)
    except Exception:
        valid_elem_mask.append(False)
        elem_polys.append(None)
        continue

    nodes = nodes[(nodes >= 0) & (nodes < n_nodes)]

    if nodes.size < 3:
        valid_elem_mask.append(False)
        elem_polys.append(None)
        continue

    coords = node_coordinates[nodes]

    try:
        poly = Polygon(coords)

        if poly.is_empty or not poly.is_valid:
            valid_elem_mask.append(False)
            elem_polys.append(None)
        else:
            valid_elem_mask.append(True)
            elem_polys.append(poly)

    except GEOSException:
        valid_elem_mask.append(False)
        elem_polys.append(None)

valid_elem_mask = np.array(valid_elem_mask, dtype=bool)

# 4) Keep only valid polygons
elem_polys = [p for p, ok in zip(elem_polys, valid_elem_mask) if ok]
depth = depth[valid_elem_mask]

# 5) Create GeoDataFrame for DFSU elements
crs_str = getattr(geom, "projection_string", None) or fallback_crs

gdf_elems = gpd.GeoDataFrame(
    {"depth": depth},
    geometry=elem_polys,
    crs=crs_str
)

# 6) Read land boundary
land = gpd.read_file(land_shp)

if land.crs is None:
    land = land.set_crs(crs_str)

if land.crs != gdf_elems.crs:
    land = land.to_crs(gdf_elems.crs)

# Dissolve land polygons into one geometry
land_diss = land.dissolve()

# 7) Clip DFSU mesh to land area
print("Clipping DFSU mesh to land boundary...")

elems_on_land = gpd.overlay(
    gdf_elems,
    land_diss,
    how="intersection",
    keep_geom_type=True
)

# 8) Calculate land area and inundated land area
elems_on_land["area_m2"] = elems_on_land.geometry.area

total_mesh_land_area_m2 = elems_on_land["area_m2"].sum()

inundated_m2 = elems_on_land.loc[
    elems_on_land["depth"] > thresh,
    "area_m2"
].sum()

# 9) Convert to km²
total_mesh_land_km2 = total_mesh_land_area_m2 / 1e6
inundated_km2 = inundated_m2 / 1e6

pct_inundated = (
    inundated_km2 / total_mesh_land_km2 * 100
    if total_mesh_land_km2 > 0
    else np.nan
)

# 10) Results
print("\nRESULTS")
print("-" * 45)
print(f"Threshold depth          : > {thresh:.2f} m")
print(f"Mesh-covered land area   : {total_mesh_land_km2:,.3f} km²")
print(f"Inundated land area      : {inundated_km2:,.3f} km²")
print(f"Percentage inundated     : {pct_inundated:.2f}%")

Available DFSU items:
0: Statistical minimum : Total water depth <Water Depth> (meter)
1: Statistical maximum : Total water depth <Water Depth> (meter)
2: Statistical mean : Total water depth <Water Depth> (meter)
Clipping DFSU mesh to land boundary...

RESULTS
---------------------------------------------
Threshold depth          : > 0.10 m
Mesh-covered land area   : 13,915.595 km²
Inundated land area      : 3,603.908 km²
Percentage inundated     : 25.90%


In [11]:
import mikeio
import geopandas as gpd
import numpy as np
from shapely.geometry import Polygon
from shapely.errors import GEOSException

# ------------------ USER INPUTS ------------------
dfsu_file = r"D:\Phd Research\Prototype HD Model\Simulation\Without Reef Scenarios\Climate change condition\100yr_compound_flood_base.m21fm - Result Files\100yr_compound_flood_base_stat.dfsu"

land_shp = r"D:\Phd Research\GIS\Shape\land_Part_area_utm.zip"

thresh = 0.10  # inundation threshold (m)
fallback_crs = "EPSG:32614"
# -------------------------------------------------

# 1) Open DFSU
ds = mikeio.open(dfsu_file)

print("Available DFSU items:")
for i, item in enumerate(ds.items):
    print(f"{i}: {item}")

# Read first item automatically
# Change items=0 if your flood-depth item is different
dset = ds.read(items=1)
geom = ds.geometry

# 2) Extract element-centered depth values
depth = dset[0].to_numpy().squeeze()

if depth.ndim != 1:
    raise RuntimeError(
        f"Unexpected depth shape: {depth.shape}. Expected 1D element values."
    )

# 3) Build element polygons
element_table = geom.element_table
node_coordinates = geom.node_coordinates[:, :2]
n_nodes = len(node_coordinates)

elem_polys = []
valid_elem_mask = []

for nodes in element_table:
    nodes = np.asarray(nodes)

    try:
        nodes = nodes.astype(int)
    except Exception:
        valid_elem_mask.append(False)
        elem_polys.append(None)
        continue

    nodes = nodes[(nodes >= 0) & (nodes < n_nodes)]

    if nodes.size < 3:
        valid_elem_mask.append(False)
        elem_polys.append(None)
        continue

    coords = node_coordinates[nodes]

    try:
        poly = Polygon(coords)

        if poly.is_empty or not poly.is_valid:
            valid_elem_mask.append(False)
            elem_polys.append(None)
        else:
            valid_elem_mask.append(True)
            elem_polys.append(poly)

    except GEOSException:
        valid_elem_mask.append(False)
        elem_polys.append(None)

valid_elem_mask = np.array(valid_elem_mask, dtype=bool)

# 4) Keep only valid polygons
elem_polys = [p for p, ok in zip(elem_polys, valid_elem_mask) if ok]
depth = depth[valid_elem_mask]

# 5) Create GeoDataFrame for DFSU elements
crs_str = getattr(geom, "projection_string", None) or fallback_crs

gdf_elems = gpd.GeoDataFrame(
    {"depth": depth},
    geometry=elem_polys,
    crs=crs_str
)

# 6) Read land boundary
land = gpd.read_file(land_shp)

if land.crs is None:
    land = land.set_crs(crs_str)

if land.crs != gdf_elems.crs:
    land = land.to_crs(gdf_elems.crs)

# Dissolve land polygons into one geometry
land_diss = land.dissolve()

# 7) Clip DFSU mesh to land area
print("Clipping DFSU mesh to land boundary...")

elems_on_land = gpd.overlay(
    gdf_elems,
    land_diss,
    how="intersection",
    keep_geom_type=True
)

# 8) Calculate land area and inundated land area
elems_on_land["area_m2"] = elems_on_land.geometry.area

total_mesh_land_area_m2 = elems_on_land["area_m2"].sum()

inundated_m2 = elems_on_land.loc[
    elems_on_land["depth"] > thresh,
    "area_m2"
].sum()

# 9) Convert to km²
total_mesh_land_km2 = total_mesh_land_area_m2 / 1e6
inundated_km2 = inundated_m2 / 1e6

pct_inundated = (
    inundated_km2 / total_mesh_land_km2 * 100
    if total_mesh_land_km2 > 0
    else np.nan
)

# 10) Results
print("\nRESULTS")
print("-" * 45)
print(f"Threshold depth          : > {thresh:.2f} m")
print(f"Mesh-covered land area   : {total_mesh_land_km2:,.3f} km²")
print(f"Inundated land area      : {inundated_km2:,.3f} km²")
print(f"Percentage inundated     : {pct_inundated:.2f}%")

Available DFSU items:
0: Statistical minimum : Total water depth <Water Depth> (meter)
1: Statistical maximum : Total water depth <Water Depth> (meter)
2: Statistical mean : Total water depth <Water Depth> (meter)
Clipping DFSU mesh to land boundary...

RESULTS
---------------------------------------------
Threshold depth          : > 0.10 m
Mesh-covered land area   : 13,915.595 km²
Inundated land area      : 13,435.019 km²
Percentage inundated     : 96.55%


In [12]:
import mikeio
import geopandas as gpd
import numpy as np
from shapely.geometry import Polygon
from shapely.errors import GEOSException

# ------------------ USER INPUTS ------------------
dfsu_file = r"D:\Phd Research\Prototype HD Model\Simulation\Without Reef Scenarios\Climate change condition\100yr_compound_flood_base_WO_VLM.m21fm - Result Files\100yr_compound_flood_base_wo_VLM_stat.dfsu"

land_shp = r"D:\Phd Research\GIS\Shape\land_Part_area_utm.zip"

thresh = 0.10  # inundation threshold (m)
fallback_crs = "EPSG:32614"
# -------------------------------------------------

# 1) Open DFSU
ds = mikeio.open(dfsu_file)

print("Available DFSU items:")
for i, item in enumerate(ds.items):
    print(f"{i}: {item}")

# Read first item automatically
# Change items=0 if your flood-depth item is different
dset = ds.read(items=1)
geom = ds.geometry

# 2) Extract element-centered depth values
depth = dset[0].to_numpy().squeeze()

if depth.ndim != 1:
    raise RuntimeError(
        f"Unexpected depth shape: {depth.shape}. Expected 1D element values."
    )

# 3) Build element polygons
element_table = geom.element_table
node_coordinates = geom.node_coordinates[:, :2]
n_nodes = len(node_coordinates)

elem_polys = []
valid_elem_mask = []

for nodes in element_table:
    nodes = np.asarray(nodes)

    try:
        nodes = nodes.astype(int)
    except Exception:
        valid_elem_mask.append(False)
        elem_polys.append(None)
        continue

    nodes = nodes[(nodes >= 0) & (nodes < n_nodes)]

    if nodes.size < 3:
        valid_elem_mask.append(False)
        elem_polys.append(None)
        continue

    coords = node_coordinates[nodes]

    try:
        poly = Polygon(coords)

        if poly.is_empty or not poly.is_valid:
            valid_elem_mask.append(False)
            elem_polys.append(None)
        else:
            valid_elem_mask.append(True)
            elem_polys.append(poly)

    except GEOSException:
        valid_elem_mask.append(False)
        elem_polys.append(None)

valid_elem_mask = np.array(valid_elem_mask, dtype=bool)

# 4) Keep only valid polygons
elem_polys = [p for p, ok in zip(elem_polys, valid_elem_mask) if ok]
depth = depth[valid_elem_mask]

# 5) Create GeoDataFrame for DFSU elements
crs_str = getattr(geom, "projection_string", None) or fallback_crs

gdf_elems = gpd.GeoDataFrame(
    {"depth": depth},
    geometry=elem_polys,
    crs=crs_str
)

# 6) Read land boundary
land = gpd.read_file(land_shp)

if land.crs is None:
    land = land.set_crs(crs_str)

if land.crs != gdf_elems.crs:
    land = land.to_crs(gdf_elems.crs)

# Dissolve land polygons into one geometry
land_diss = land.dissolve()

# 7) Clip DFSU mesh to land area
print("Clipping DFSU mesh to land boundary...")

elems_on_land = gpd.overlay(
    gdf_elems,
    land_diss,
    how="intersection",
    keep_geom_type=True
)

# 8) Calculate land area and inundated land area
elems_on_land["area_m2"] = elems_on_land.geometry.area

total_mesh_land_area_m2 = elems_on_land["area_m2"].sum()

inundated_m2 = elems_on_land.loc[
    elems_on_land["depth"] > thresh,
    "area_m2"
].sum()

# 9) Convert to km²
total_mesh_land_km2 = total_mesh_land_area_m2 / 1e6
inundated_km2 = inundated_m2 / 1e6

pct_inundated = (
    inundated_km2 / total_mesh_land_km2 * 100
    if total_mesh_land_km2 > 0
    else np.nan
)

# 10) Results
print("\nRESULTS")
print("-" * 45)
print(f"Threshold depth          : > {thresh:.2f} m")
print(f"Mesh-covered land area   : {total_mesh_land_km2:,.3f} km²")
print(f"Inundated land area      : {inundated_km2:,.3f} km²")
print(f"Percentage inundated     : {pct_inundated:.2f}%")

Available DFSU items:
0: Statistical minimum : Total water depth <Water Depth> (meter)
1: Statistical maximum : Total water depth <Water Depth> (meter)
2: Statistical mean : Total water depth <Water Depth> (meter)
Clipping DFSU mesh to land boundary...

RESULTS
---------------------------------------------
Threshold depth          : > 0.10 m
Mesh-covered land area   : 13,915.595 km²
Inundated land area      : 13,328.578 km²
Percentage inundated     : 95.78%


In [13]:
import mikeio
import geopandas as gpd
import numpy as np
from shapely.geometry import Polygon
from shapely.errors import GEOSException

# ------------------ USER INPUTS ------------------
dfsu_file = r"D:\Phd Research\Prototype HD Model\Simulation\Without Reef Scenarios\Climate change condition\5yr_Compound_Flood_wo_VLM.m21fm - Result Files\5yr_compound_flood_wo_VLM_stat.dfsu"

land_shp = r"D:\Phd Research\GIS\Shape\land_Part_area_utm.zip"

thresh = 0.10  # inundation threshold (m)
fallback_crs = "EPSG:32614"
# -------------------------------------------------

# 1) Open DFSU
ds = mikeio.open(dfsu_file)

print("Available DFSU items:")
for i, item in enumerate(ds.items):
    print(f"{i}: {item}")

# Read first item automatically
# Change items=0 if your flood-depth item is different
dset = ds.read(items=1)
geom = ds.geometry

# 2) Extract element-centered depth values
depth = dset[0].to_numpy().squeeze()

if depth.ndim != 1:
    raise RuntimeError(
        f"Unexpected depth shape: {depth.shape}. Expected 1D element values."
    )

# 3) Build element polygons
element_table = geom.element_table
node_coordinates = geom.node_coordinates[:, :2]
n_nodes = len(node_coordinates)

elem_polys = []
valid_elem_mask = []

for nodes in element_table:
    nodes = np.asarray(nodes)

    try:
        nodes = nodes.astype(int)
    except Exception:
        valid_elem_mask.append(False)
        elem_polys.append(None)
        continue

    nodes = nodes[(nodes >= 0) & (nodes < n_nodes)]

    if nodes.size < 3:
        valid_elem_mask.append(False)
        elem_polys.append(None)
        continue

    coords = node_coordinates[nodes]

    try:
        poly = Polygon(coords)

        if poly.is_empty or not poly.is_valid:
            valid_elem_mask.append(False)
            elem_polys.append(None)
        else:
            valid_elem_mask.append(True)
            elem_polys.append(poly)

    except GEOSException:
        valid_elem_mask.append(False)
        elem_polys.append(None)

valid_elem_mask = np.array(valid_elem_mask, dtype=bool)

# 4) Keep only valid polygons
elem_polys = [p for p, ok in zip(elem_polys, valid_elem_mask) if ok]
depth = depth[valid_elem_mask]

# 5) Create GeoDataFrame for DFSU elements
crs_str = getattr(geom, "projection_string", None) or fallback_crs

gdf_elems = gpd.GeoDataFrame(
    {"depth": depth},
    geometry=elem_polys,
    crs=crs_str
)

# 6) Read land boundary
land = gpd.read_file(land_shp)

if land.crs is None:
    land = land.set_crs(crs_str)

if land.crs != gdf_elems.crs:
    land = land.to_crs(gdf_elems.crs)

# Dissolve land polygons into one geometry
land_diss = land.dissolve()

# 7) Clip DFSU mesh to land area
print("Clipping DFSU mesh to land boundary...")

elems_on_land = gpd.overlay(
    gdf_elems,
    land_diss,
    how="intersection",
    keep_geom_type=True
)

# 8) Calculate land area and inundated land area
elems_on_land["area_m2"] = elems_on_land.geometry.area

total_mesh_land_area_m2 = elems_on_land["area_m2"].sum()

inundated_m2 = elems_on_land.loc[
    elems_on_land["depth"] > thresh,
    "area_m2"
].sum()

# 9) Convert to km²
total_mesh_land_km2 = total_mesh_land_area_m2 / 1e6
inundated_km2 = inundated_m2 / 1e6

pct_inundated = (
    inundated_km2 / total_mesh_land_km2 * 100
    if total_mesh_land_km2 > 0
    else np.nan
)

# 10) Results
print("\nRESULTS")
print("-" * 45)
print(f"Threshold depth          : > {thresh:.2f} m")
print(f"Mesh-covered land area   : {total_mesh_land_km2:,.3f} km²")
print(f"Inundated land area      : {inundated_km2:,.3f} km²")
print(f"Percentage inundated     : {pct_inundated:.2f}%")

Available DFSU items:
0: Statistical minimum : Total water depth <Water Depth> (meter)
1: Statistical maximum : Total water depth <Water Depth> (meter)
2: Statistical mean : Total water depth <Water Depth> (meter)
Clipping DFSU mesh to land boundary...

RESULTS
---------------------------------------------
Threshold depth          : > 0.10 m
Mesh-covered land area   : 13,915.595 km²
Inundated land area      : 3,385.008 km²
Percentage inundated     : 24.33%


In [10]:
import mikeio
import geopandas as gpd
import numpy as np
from shapely.geometry import Polygon
from shapely.errors import GEOSException

# ------------------ USER INPUTS ------------------
dfsu_file = r"D:\Phd Research\Prototype HD Model\Simulation\Without Reef Scenarios\Climate change condition\100yr_compound_flood_base_10_km_seaward.m21fm - Result Files\100yr_compound_flood_base_10km_seaward_2D_Stat.dfsu"

land_shp = r"D:\Phd Research\GIS\Shape\land_Part_area_utm.zip"

thresh = 0.10  # inundation threshold (m)
fallback_crs = "EPSG:32614"
# -------------------------------------------------

# 1) Open DFSU
ds = mikeio.open(dfsu_file)

print("Available DFSU items:")
for i, item in enumerate(ds.items):
    print(f"{i}: {item}")

# Read first item automatically
# Change items=0 if your flood-depth item is different
dset = ds.read(items=1)
geom = ds.geometry

# 2) Extract element-centered depth values
depth = dset[0].to_numpy().squeeze()

if depth.ndim != 1:
    raise RuntimeError(
        f"Unexpected depth shape: {depth.shape}. Expected 1D element values."
    )

# 3) Build element polygons
element_table = geom.element_table
node_coordinates = geom.node_coordinates[:, :2]
n_nodes = len(node_coordinates)

elem_polys = []
valid_elem_mask = []

for nodes in element_table:
    nodes = np.asarray(nodes)

    try:
        nodes = nodes.astype(int)
    except Exception:
        valid_elem_mask.append(False)
        elem_polys.append(None)
        continue

    nodes = nodes[(nodes >= 0) & (nodes < n_nodes)]

    if nodes.size < 3:
        valid_elem_mask.append(False)
        elem_polys.append(None)
        continue

    coords = node_coordinates[nodes]

    try:
        poly = Polygon(coords)

        if poly.is_empty or not poly.is_valid:
            valid_elem_mask.append(False)
            elem_polys.append(None)
        else:
            valid_elem_mask.append(True)
            elem_polys.append(poly)

    except GEOSException:
        valid_elem_mask.append(False)
        elem_polys.append(None)

valid_elem_mask = np.array(valid_elem_mask, dtype=bool)

# 4) Keep only valid polygons
elem_polys = [p for p, ok in zip(elem_polys, valid_elem_mask) if ok]
depth = depth[valid_elem_mask]

# 5) Create GeoDataFrame for DFSU elements
crs_str = getattr(geom, "projection_string", None) or fallback_crs

gdf_elems = gpd.GeoDataFrame(
    {"depth": depth},
    geometry=elem_polys,
    crs=crs_str
)

# 6) Read land boundary
land = gpd.read_file(land_shp)

if land.crs is None:
    land = land.set_crs(crs_str)

if land.crs != gdf_elems.crs:
    land = land.to_crs(gdf_elems.crs)

# Dissolve land polygons into one geometry
land_diss = land.dissolve()

# 7) Clip DFSU mesh to land area
print("Clipping DFSU mesh to land boundary...")

elems_on_land = gpd.overlay(
    gdf_elems,
    land_diss,
    how="intersection",
    keep_geom_type=True
)

# 8) Calculate land area and inundated land area
elems_on_land["area_m2"] = elems_on_land.geometry.area

total_mesh_land_area_m2 = elems_on_land["area_m2"].sum()

inundated_m2 = elems_on_land.loc[
    elems_on_land["depth"] > thresh,
    "area_m2"
].sum()

# 9) Convert to km²
total_mesh_land_km2 = total_mesh_land_area_m2 / 1e6
inundated_km2 = inundated_m2 / 1e6

pct_inundated = (
    inundated_km2 / total_mesh_land_km2 * 100
    if total_mesh_land_km2 > 0
    else np.nan
)

# 10) Results
print("\nRESULTS")
print("-" * 45)
print(f"Threshold depth          : > {thresh:.2f} m")
print(f"Mesh-covered land area   : {total_mesh_land_km2:,.3f} km²")
print(f"Inundated land area      : {inundated_km2:,.3f} km²")
print(f"Percentage inundated     : {pct_inundated:.2f}%")

Available DFSU items:
0: Statistical minimum : Total water depth <Water Depth> (meter)
1: Statistical maximum : Total water depth <Water Depth> (meter)
2: Statistical mean : Total water depth <Water Depth> (meter)
Clipping DFSU mesh to land boundary...

RESULTS
---------------------------------------------
Threshold depth          : > 0.10 m
Mesh-covered land area   : 13,915.595 km²
Inundated land area      : 13,313.821 km²
Percentage inundated     : 95.68%


In [8]:
import mikeio
import geopandas as gpd
import numpy as np
from shapely.geometry import Polygon
from shapely.errors import GEOSException

# ------------------ USER INPUTS ------------------
dfsu_file = r"D:\Phd Research\Prototype HD Model\Simulation\Without Reef Scenarios\Climate change condition\100yr_compound_flood_base_10_km_landward.m21fm - Result Files\100yr_compound_flood_base_10km_landward_2D_Stat.dfsu"
land_shp = r"D:\Phd Research\GIS\Shape\land_Part_area_utm.zip"

thresh = 0.10  # inundation threshold (m)
fallback_crs = "EPSG:32614"
# -------------------------------------------------

# 1) Open DFSU
ds = mikeio.open(dfsu_file)

print("Available DFSU items:")
for i, item in enumerate(ds.items):
    print(f"{i}: {item}")

# Read first item automatically
# Change items=0 if your flood-depth item is different
dset = ds.read(items=1)
geom = ds.geometry

# 2) Extract element-centered depth values
depth = dset[0].to_numpy().squeeze()

if depth.ndim != 1:
    raise RuntimeError(
        f"Unexpected depth shape: {depth.shape}. Expected 1D element values."
    )

# 3) Build element polygons
element_table = geom.element_table
node_coordinates = geom.node_coordinates[:, :2]
n_nodes = len(node_coordinates)

elem_polys = []
valid_elem_mask = []

for nodes in element_table:
    nodes = np.asarray(nodes)

    try:
        nodes = nodes.astype(int)
    except Exception:
        valid_elem_mask.append(False)
        elem_polys.append(None)
        continue

    nodes = nodes[(nodes >= 0) & (nodes < n_nodes)]

    if nodes.size < 3:
        valid_elem_mask.append(False)
        elem_polys.append(None)
        continue

    coords = node_coordinates[nodes]

    try:
        poly = Polygon(coords)

        if poly.is_empty or not poly.is_valid:
            valid_elem_mask.append(False)
            elem_polys.append(None)
        else:
            valid_elem_mask.append(True)
            elem_polys.append(poly)

    except GEOSException:
        valid_elem_mask.append(False)
        elem_polys.append(None)

valid_elem_mask = np.array(valid_elem_mask, dtype=bool)

# 4) Keep only valid polygons
elem_polys = [p for p, ok in zip(elem_polys, valid_elem_mask) if ok]
depth = depth[valid_elem_mask]

# 5) Create GeoDataFrame for DFSU elements
crs_str = getattr(geom, "projection_string", None) or fallback_crs

gdf_elems = gpd.GeoDataFrame(
    {"depth": depth},
    geometry=elem_polys,
    crs=crs_str
)

# 6) Read land boundary
land = gpd.read_file(land_shp)

if land.crs is None:
    land = land.set_crs(crs_str)

if land.crs != gdf_elems.crs:
    land = land.to_crs(gdf_elems.crs)

# Dissolve land polygons into one geometry
land_diss = land.dissolve()

# 7) Clip DFSU mesh to land area
print("Clipping DFSU mesh to land boundary...")

elems_on_land = gpd.overlay(
    gdf_elems,
    land_diss,
    how="intersection",
    keep_geom_type=True
)

# 8) Calculate land area and inundated land area
elems_on_land["area_m2"] = elems_on_land.geometry.area

total_mesh_land_area_m2 = elems_on_land["area_m2"].sum()

inundated_m2 = elems_on_land.loc[
    elems_on_land["depth"] > thresh,
    "area_m2"
].sum()

# 9) Convert to km²
total_mesh_land_km2 = total_mesh_land_area_m2 / 1e6
inundated_km2 = inundated_m2 / 1e6

pct_inundated = (
    inundated_km2 / total_mesh_land_km2 * 100
    if total_mesh_land_km2 > 0
    else np.nan
)

# 10) Results
print("\nRESULTS")
print("-" * 45)
print(f"Threshold depth          : > {thresh:.2f} m")
print(f"Mesh-covered land area   : {total_mesh_land_km2:,.3f} km²")
print(f"Inundated land area      : {inundated_km2:,.3f} km²")
print(f"Percentage inundated     : {pct_inundated:.2f}%")

Available DFSU items:
0: Statistical minimum : Total water depth <Water Depth> (meter)
1: Statistical maximum : Total water depth <Water Depth> (meter)
2: Statistical mean : Total water depth <Water Depth> (meter)
Clipping DFSU mesh to land boundary...

RESULTS
---------------------------------------------
Threshold depth          : > 0.10 m
Mesh-covered land area   : 13,914.970 km²
Inundated land area      : 13,285.188 km²
Percentage inundated     : 95.47%
